# 📸 Module 1.1: What Is An Image?

## From Photons to Pixels — The Complete Mathematical Journey

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_01_Image_Foundations/01_What_Is_An_Image/what_is_an_image.ipynb)

---

## 🎯 Learning Objectives
1. Understand what an image truly is (physically and mathematically)
2. Represent images as continuous functions and discrete matrices
3. Visualize every step of image formation
4. Connect image representation to why RL can manipulate images

---

## 1. Physical Definition of an Image

### An image is a 2D function of light intensity:

$$f(x, y) = i(x, y) \cdot r(x, y)$$

Where:
- $f(x, y)$ = intensity at point $(x, y)$
- $i(x, y)$ = illumination component (light source)
- $r(x, y)$ = reflectance component (object surface)

**Constraints:**
$$0 < i(x,y) < \infty$$
$$0 < r(x,y) < 1$$

### Why This Matters for RL:
An RL agent that processes images is essentially learning to understand and manipulate this function $f(x,y)$. The **state** in our RL framework will be the image itself!

In [ ]:
# Setup - Install dependencies (Google Colab)
!pip install numpy matplotlib opencv-python-headless pillow scikit-image -q

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
import cv2
from PIL import Image
import urllib.request
import os

plt.style.use('default')
print("✅ All libraries loaded successfully!")

## Deep Mathematical Derivation: Image Formation

### Step 1: Electromagnetic Radiation to Intensity
Light intensity at wavelength $\lambda$ hitting sensor at position $(x,y)$:
$$E(x, y, \lambda) = \int_{\text{scene}} L(x', y', \lambda) \cdot \text{BRDF}(x', y', \omega_i, \omega_o) \cdot \cos\theta \, d\omega$$

### Step 2: Sensor Integration (Continuous → Discrete)
The sensor integrates over exposure time $T$ and pixel area $A$:
$$I_{\text{continuous}}(x, y) = \frac{1}{T} \int_0^T \int_A E(x+u, y+v, t) \, du\, dv\, dt$$

### Step 3: Sampling Theorem (Nyquist-Shannon)
To perfectly reconstruct $f(x,y)$ from samples, sampling frequency must satisfy:
$$f_s > 2 f_{\max}$$

**Proof:** If $f(x)$ is bandlimited with $F(\omega) = 0$ for $|\omega| > \omega_{\max}$, then:
$$f(x) = \sum_{n=-\infty}^{\infty} f(n/f_s) \cdot \text{sinc}\left(\frac{x - n/f_s}{1/f_s}\right)$$

**Derivation step-by-step:**
1. Sample: $f_s[n] = f(n \cdot \Delta x)$ where $\Delta x = 1/f_s$
2. In frequency domain: $F_s(\omega) = \frac{1}{\Delta x} \sum_{k=-\infty}^{\infty} F(\omega - k/\Delta x)$
3. If $f_s > 2f_{\max}$: copies don't overlap → perfect reconstruction
4. If $f_s \leq 2f_{\max}$: aliasing occurs → information lost!

### Step 4: Quantization Error Analysis
For uniform quantization with $L = 2^k$ levels:
$$Q(x) = \Delta \cdot \left\lfloor \frac{x}{\Delta} + \frac{1}{2} \right\rfloor, \quad \Delta = \frac{x_{\max} - x_{\min}}{L-1}$$

**Quantization error:** $e = x - Q(x)$, uniformly distributed on $[-\Delta/2, \Delta/2]$

**SQNR (Signal-to-Quantization-Noise Ratio):**
$$\text{SQNR} = 10 \log_{10}\left(\frac{\sigma_x^2}{\sigma_e^2}\right) = 10 \log_{10}\left(\frac{12 \sigma_x^2}{\Delta^2}\right) \approx 6.02k + 1.76 \text{ dB}$$

**Derivation:**
- $\sigma_e^2 = E[e^2] = \int_{-\Delta/2}^{\Delta/2} e^2 \cdot \frac{1}{\Delta} de = \frac{\Delta^2}{12}$
- For $k$ bits: $\Delta = \frac{1}{2^k - 1} \approx 2^{-k}$
- Therefore: $\text{SQNR} \approx 6.02k$ dB (each bit adds ~6 dB)

### Step 5: Why This Matters for RL
The **state space size** of an image-based RL problem:
$$|S| = L^{H \times W \times C} = (2^k)^{H \times W \times C}$$

For 8-bit RGB 32×32 image: $|S| = 256^{3072} \approx 10^{7395}$ — intractable for tabular RL!

**This is WHY we need Deep RL (function approximation) for image-based agents.**

In [ ]:
# Download REAL open-source dataset
from skimage import data
import torchvision
import torchvision.transforms as transforms

# Real images from scikit-image (built-in, no download needed)
astronaut_img = data.astronaut()       # 512x512x3 real photo
camera_img = data.camera()             # 512x512 grayscale real photo  
coins_img = data.coins()               # Real coins photograph
coffee_img = data.coffee()             # 400x600x3 real photo

# MNIST - 70,000 real handwritten digits (auto-downloads ~11MB)
mnist_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True)
print(f"✅ MNIST loaded: {len(mnist_dataset)} real handwritten digit images (28x28)")
print(f"✅ scikit-image loaded: astronaut {astronaut_img.shape}, camera {camera_img.shape}")

## 2. Creating Our First Image from Scratch

Let's build an image mathematically — a 2D Gaussian function:

$$f(x, y) = A \cdot \exp\left(-\frac{(x-x_0)^2}{2\sigma_x^2} - \frac{(y-y_0)^2}{2\sigma_y^2}\right)$$

In [ ]:
# Create a continuous 2D function (image)
x = np.linspace(-5, 5, 256)
y = np.linspace(-5, 5, 256)
X, Y = np.meshgrid(x, y)

# 2D Gaussian - this IS an image mathematically
sigma_x, sigma_y = 1.5, 1.5
x0, y0 = 0, 0
A = 255  # Maximum intensity

Z = A * np.exp(-((X - x0)**2 / (2 * sigma_x**2) + (Y - y0)**2 / (2 * sigma_y**2)))

# Visualize as both 3D surface and 2D image
fig = plt.figure(figsize=(16, 6))

# 3D Surface - showing image as a function
ax1 = fig.add_subplot(121, projection='3d')
ax1.plot_surface(X, Y, Z, cmap='hot', alpha=0.8)
ax1.set_xlabel('X position')
ax1.set_ylabel('Y position')
ax1.set_zlabel('Intensity f(x,y)')
ax1.set_title('Image as 3D Function\n$f(x,y) = A \cdot e^{-(x^2+y^2)/2\sigma^2}$')

# 2D Image - what we actually see
ax2 = fig.add_subplot(122)
im = ax2.imshow(Z, cmap='hot', origin='lower')
ax2.set_title('Same Function as 2D Image\n(What our eyes see)')
ax2.set_xlabel('X pixels')
ax2.set_ylabel('Y pixels')
plt.colorbar(im, ax=ax2, label='Intensity')

plt.tight_layout()
plt.savefig('image_as_function.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📊 Image Shape: {Z.shape}")
print(f"📊 Min Intensity: {Z.min():.2f}")
print(f"📊 Max Intensity: {Z.max():.2f}")
print(f"📊 Data Type: {Z.dtype}")

## The Point Spread Function — How Optical Systems Form Images

Real cameras do not capture the ideal image $f(x,y)$ — the lens introduces blur modeled by the Point Spread Function (PSF).

### Step 1: Image Formation Model

The captured image is the convolution of the ideal image with the PSF:

$$g(x, y) = f(x, y) * h(x, y) + \eta(x, y)$$

where $h(x, y)$ is the PSF (impulse response of the optical system) and $\eta$ is sensor noise.

### Step 2: PSF of a Diffraction-Limited Lens

For a circular aperture of diameter $D$ at focal length $f$:

$$h(r) = \left[\frac{2J_1(\pi r D / \lambda f)}{\pi r D / \lambda f}\right]^2$$

where $J_1$ is the first-order Bessel function and $\lambda$ is the wavelength. This is the **Airy pattern**.

The first zero occurs at $r_0 = 1.22 \lambda f / D$ (the Rayleigh resolution limit).

### Step 3: Out-of-Focus Blur as a Pillbox Function

For a lens focused at distance $d_f$ imaging a point at distance $d$:

$$h(x, y) = \begin{cases} 1/(\pi R^2) & x^2 + y^2 \leq R^2 \\ 0 & \text{otherwise} \end{cases}$$

where $R$ is the blur circle radius:
$$R = \frac{D}{2} \cdot \frac{|d - d_f|}{d} \cdot \frac{f}{d_f - f}$$

### Step 4: Motion Blur as a Line PSF

Camera motion during exposure creates a line-shaped PSF:

$$h(x, y) = \begin{cases} 1/L & \text{if } x \cos\theta + y\sin\theta \in [0, L] \text{ and } -x\sin\theta + y\cos\theta = 0 \\ 0 & \text{otherwise} \end{cases}$$

where $L$ is the motion length and $\theta$ is the motion direction.

### Step 5: Image Deconvolution (Inverse Problem)

In the frequency domain: $G(u,v) = F(u,v) \cdot H(u,v) + N(u,v)$

**Inverse filtering:** $\hat{F} = G / H$ — fails where $H \approx 0$ (noise amplification)

**Wiener deconvolution:** $\hat{F} = \frac{H^*}{|H|^2 + \sigma_n^2/\sigma_f^2} \cdot G$ — regularized, stable

**RL connection:** An RL agent can learn to estimate and correct for the PSF adaptively:
- State: blurred image + estimated blur kernel
- Action: deconvolution parameters (regularization strength, kernel estimate)
- Reward: sharpness metric (e.g., total variation of the result)

## 3. Digital Image — Sampling and Quantization

A digital image is created through two processes:

### Sampling (Spatial Discretization)
$$f_s(m, n) = f(m \cdot \Delta x, \; n \cdot \Delta y)$$

Where $\Delta x, \Delta y$ are the sampling intervals.

### Quantization (Intensity Discretization)
$$f_q(m, n) = \text{round}\left(\frac{f_s(m,n)}{\Delta I}\right) \cdot \Delta I$$

Where $\Delta I = \frac{I_{max}}{2^k - 1}$ for $k$-bit quantization.

**Result:** A matrix $\mathbf{I} \in \mathbb{R}^{M \times N}$ where each element $I[m,n] \in \{0, 1, ..., 2^k-1\}$

## The 2D Discrete Fourier Transform — Frequency Analysis of Images

Every image can be decomposed into a sum of sinusoidal patterns at different frequencies and orientations. The DFT provides this decomposition.

### Definition: 2D DFT

For an image $I[m, n]$ of size $M \times N$, the 2D DFT is:

$$F[u, v] = \sum_{m=0}^{M-1} \sum_{n=0}^{N-1} I[m, n] \cdot e^{-j2\pi\left(\frac{um}{M} + \frac{vn}{N}\right)}$$

Each coefficient $F[u, v]$ represents the amplitude and phase of a 2D sinusoidal basis function with horizontal frequency $u/M$ and vertical frequency $v/N$.

### Key Properties (with proofs)

**1. Linearity:** $\mathcal{F}\{aI_1 + bI_2\} = a\mathcal{F}\{I_1\} + b\mathcal{F}\{I_2\}$

*Proof:* Follows directly from linearity of summation. $\blacksquare$

**2. Parseval's Theorem (Energy Conservation):**
$$\sum_{m,n} |I[m,n]|^2 = \frac{1}{MN}\sum_{u,v} |F[u,v]|^2$$

*Proof:* 
$$\sum_{m,n} |I[m,n]|^2 = \sum_{m,n} I[m,n] \cdot I^*[m,n]$$
$$= \sum_{m,n} I[m,n] \cdot \frac{1}{MN}\sum_{u,v} F^*[u,v] e^{-j2\pi(um/M + vn/N)}$$
$$= \frac{1}{MN}\sum_{u,v} F^*[u,v] \underbrace{\sum_{m,n} I[m,n] e^{-j2\pi(um/M + vn/N)}}_{= F[u,v]} = \frac{1}{MN}\sum_{u,v} |F[u,v]|^2 \quad \blacksquare$$

**3. Shift Theorem:** Shifting the image by $(m_0, n_0)$ multiplies the spectrum by a phase factor:
$$\mathcal{F}\{I[m - m_0, n - n_0]\} = F[u,v] \cdot e^{-j2\pi(um_0/M + vn_0/N)}$$

This means translation does NOT change the magnitude spectrum $|F[u,v]|$ — only the phase changes.

### Frequency Interpretation for Images

- **$F[0, 0] = \sum_{m,n} I[m,n]$**: the DC component (proportional to mean brightness)
- **Low frequencies** (small $u, v$): smooth, slowly-varying regions
- **High frequencies** (large $u, v$): edges, textures, noise
- **Magnitude** $|F[u,v]|$: "how much" of each frequency
- **Phase** $\angle F[u,v]$: "where" each frequency component is located

**The phase carries more perceptual information than magnitude** — swapping phases between two images makes the result look like the image whose phase was used, not whose magnitude was used.

## Aliasing — Mathematical Condition and Visual Consequences

Aliasing is one of the most important artifacts in digital imaging. It occurs when sampling violates the Nyquist condition, causing high-frequency content to masquerade as low-frequency patterns.

### The Nyquist-Shannon Sampling Theorem (Formal Statement)

**Theorem:** A continuous signal $f(x)$ that is bandlimited with maximum frequency $f_{\max}$ (i.e., $\hat{f}(\omega) = 0$ for $|\omega| > 2\pi f_{\max}$) can be perfectly reconstructed from its samples $f(n/f_s)$ if and only if:

$$f_s > 2 f_{\max}$$

The critical frequency $f_N = f_s / 2$ is called the **Nyquist frequency**.

### Proof of Aliasing

When a signal with frequency $f_0 > f_N$ is sampled at rate $f_s$, it appears identical to a lower-frequency signal:

$$\sin(2\pi f_0 \cdot n/f_s) = \sin(2\pi (f_0 - kf_s) \cdot n/f_s)$$

for any integer $k$, because $\sin(2\pi f_0 n/f_s - 2\pi kn) = \sin(2\pi f_0 n/f_s)$ since $2\pi kn$ is a multiple of $2\pi$ for integer $n$.

The aliased frequency is: $f_{\text{alias}} = f_0 \mod f_s$, folded back into $[0, f_N]$:

$$f_{\text{alias}} = \begin{cases} f_0 \mod f_s & \text{if } (f_0 \mod f_s) \leq f_N \\ f_s - (f_0 \mod f_s) & \text{if } (f_0 \mod f_s) > f_N \end{cases}$$

$\blacksquare$

### 2D Aliasing in Images

For a 2D image sampled on a grid with spacing $\Delta x, \Delta y$:
- Maximum representable frequency: $f_{x,\max} = 1/(2\Delta x)$, $f_{y,\max} = 1/(2\Delta y)$
- Any spatial pattern with frequency above these limits aliases

**Common visual artifacts:**
- **Moiré patterns:** Repeated fine structures (brick walls, fabric) beat against the pixel grid
- **Jagged edges (jaggies):** Diagonal lines at non-45° angles show staircase patterns
- **Wagon-wheel effect:** Fine radial patterns appear to rotate backwards

### Anti-Aliasing — The Solution

**Pre-filtering:** Apply a low-pass filter before sampling to remove frequencies above $f_N$:

$$f_{\text{anti-aliased}}(x) = (f * G_\sigma)(x) \quad \text{where } \sigma \geq \frac{1}{2f_N}$$

This is why cameras have anti-aliasing (optical low-pass) filters — they intentionally blur the image slightly before the sensor samples it, trading sharpness for artifact-free sampling.

### RL Implication

When an RL agent downsamples images (common for reducing state space size), it MUST anti-alias first, or the aliased high-frequency content creates false features that confuse the agent's policy.

In [ ]:
# Demonstrate sampling at different resolutions
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

resolutions = [4, 8, 16, 32, 64, 128, 256, 512]

for idx, res in enumerate(resolutions):
    row = idx // 4
    col = idx % 4
    
    # Sample the continuous function at different resolutions
    x_s = np.linspace(-5, 5, res)
    y_s = np.linspace(-5, 5, res)
    X_s, Y_s = np.meshgrid(x_s, y_s)
    Z_s = A * np.exp(-((X_s)**2 / (2 * sigma_x**2) + (Y_s)**2 / (2 * sigma_y**2)))
    
    axes[row, col].imshow(Z_s, cmap='hot', interpolation='nearest')
    axes[row, col].set_title(f'{res}×{res} pixels\n({res*res} samples)')
    axes[row, col].axis('off')

plt.suptitle('Sampling: Same Image at Different Resolutions\n'
             '(More samples = More detail)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('sampling_demo.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Demonstrate quantization at different bit depths
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

bit_depths = [1, 2, 3, 4, 5, 6, 7, 8]
continuous_image = Z.copy()

for idx, bits in enumerate(bit_depths):
    row = idx // 4
    col = idx % 4
    
    # Quantize to k bits
    levels = 2**bits
    quantized = np.round(continuous_image / 255 * (levels - 1)) * (255 / (levels - 1))
    
    axes[row, col].imshow(quantized, cmap='gray', vmin=0, vmax=255)
    axes[row, col].set_title(f'{bits}-bit ({levels} levels)')
    axes[row, col].axis('off')

plt.suptitle('Quantization: Same Image at Different Bit Depths\n'
             '(More bits = More intensity levels)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('quantization_demo.png', dpi=150, bbox_inches='tight')
plt.show()

## Bayer Pattern and Demosaicing — The Mathematics of Color Reconstruction

Almost every digital camera captures color using a single sensor with a color filter array (CFA). Reconstructing the full color image requires solving an interpolation problem.

### The Bayer Pattern

The Bayer CFA arranges color filters in a repeating 2×2 pattern:
$$\begin{bmatrix} G & R \\ B & G \end{bmatrix}$$

**Sampling ratios:** 50% Green, 25% Red, 25% Blue (green is doubled because the human eye is most sensitive to green wavelengths, matching the luminous efficiency function $V(\lambda)$).

### The Demosaicing Problem

At each pixel, only ONE color channel is measured. The other two must be interpolated:
- At a Red pixel: $R$ is known, $G$ and $B$ must be estimated
- At a Green pixel: $G$ is known, $R$ and $B$ must be estimated  
- At a Blue pixel: $B$ is known, $R$ and $G$ must be estimated

### Bilinear Demosaicing (Simplest Method)

Estimate missing values by averaging known neighbors:

**Green at a Red pixel** $(i, j)$:
$$\hat{G}(i,j) = \frac{G(i-1,j) + G(i+1,j) + G(i,j-1) + G(i,j+1)}{4}$$

**Red at a Green pixel** (in a red row):
$$\hat{R}(i,j) = \frac{R(i,j-1) + R(i,j+1)}{2}$$

**Red at a Blue pixel** (diagonal neighbors):
$$\hat{R}(i,j) = \frac{R(i-1,j-1) + R(i-1,j+1) + R(i+1,j-1) + R(i+1,j+1)}{4}$$

### Frequency-Domain Analysis of Demosaicing

The Bayer pattern can be modeled as modulation: the raw CFA image is the sum of baseband (luminance) and modulated (chrominance) signals:

$$I_{\text{CFA}}(x,y) = L(x,y) + C_1(x,y)\cos(\pi x) + C_2(x,y)\cos(\pi y) + C_3(x,y)\cos(\pi x)\cos(\pi y)$$

**Demosaicing = demodulation:** Separating these components is equivalent to bandpass filtering in the frequency domain. Aliasing occurs when luminance and chrominance spectra overlap — this causes the color moiré artifacts visible in fine textures.

### Why This Matters

Understanding the Bayer pattern and demosaicing is important for RL-based image processing because:
1. Raw camera data has different noise statistics than processed images (Poisson in raw vs. correlated Gaussian after demosaicing)
2. Processing raw data directly can give RL agents access to more information
3. The demosaicing artifacts are a source of learnable patterns that the agent must handle

## Image as a Random Field — Statistical Models of Natural Images

Treating images as random fields provides the mathematical framework for noise analysis, texture modeling, and optimal filtering — all essential for RL-based image processing.

### Step 1: Stationary Random Fields

A random field $I(x, y)$ is **wide-sense stationary** (WSS) if:
1. $E[I(x, y)] = \mu$ (constant mean)
2. $\text{Cov}[I(x_1, y_1), I(x_2, y_2)] = R(x_1 - x_2, y_1 - y_2)$ (covariance depends only on displacement)

**Consequence:** The power spectral density (PSD) is well-defined:
$$S(u, v) = \mathcal{F}\{R(\Delta x, \Delta y)\}$$

### Step 2: The $1/f^2$ Power Law of Natural Images

**Empirical finding:** The radially averaged PSD of natural images follows:
$$S(f) \propto f^{-\beta}, \quad \beta \approx 2, \quad f = \sqrt{u^2 + v^2}$$

**Physical interpretation:** Natural scenes are approximately scale-invariant — zooming in reveals the same statistical structure. The $1/f^2$ law is the Fourier-domain manifestation of this self-similarity.

**Derivation from scale invariance:** If $I(\lambda x, \lambda y) \stackrel{d}{=} \lambda^H I(x, y)$ for some Hurst exponent $H$, then the PSD scales as:
$$S(\lambda f) = \lambda^{-(2H+2)} S(f) \implies S(f) \propto f^{-(2H+2)}$$

For natural images, $H \approx 0$, giving $\beta = 2H + 2 \approx 2$. $\blacksquare$

### Step 3: Implications for Compression and Denoising

The $1/f^2$ power law means:
- Most energy is in low frequencies (smooth regions dominate)
- High-frequency content (edges, textures) has low energy but is perceptually important
- **Compression:** Allocate more bits to low frequencies (where the energy is)
- **Denoising:** White noise ($S_N = \text{const}$) dominates at high frequencies → high-frequency suppression is effective

### Step 4: Whitening Transform

The whitening transform flattens the power spectrum to uniform:
$$I_{\text{white}}(x, y) = \mathcal{F}^{-1}\left\{\frac{\hat{I}(u,v)}{\sqrt{S(u,v)}}\right\}$$

**Result:** $S_{\text{white}}(u,v) = 1$ (flat spectrum — all frequencies equally represented).

**Biological relevance:** The retina performs approximate whitening through center-surround receptive fields, emphasizing edges (high frequencies) relative to smooth regions — making the neural representation more efficient.

### RL Connection

For RL agents, the $1/f^2$ prior tells us:
- The state space has predictable structure (low-rank)
- Random image perturbations (exploration) should respect the power law
- Feature extraction should emphasize high frequencies (where the discriminative info is, relative to energy)

## 4. Loading a Real Image — Understanding the Matrix

Every digital image is stored as a matrix (or tensor for color images):

$$\mathbf{I} = \begin{bmatrix} I[0,0] & I[0,1] & \cdots & I[0,N-1] \\ I[1,0] & I[1,1] & \cdots & I[1,N-1] \\ \vdots & \vdots & \ddots & \vdots \\ I[M-1,0] & I[M-1,1] & \cdots & I[M-1,N-1] \end{bmatrix}$$

In [ ]:
# Create a small sample image for detailed exploration
# Using a simple gradient + shape image
def create_sample_image(size=64):
    """Create a sample image with known properties for study."""
    img = np.zeros((size, size, 3), dtype=np.uint8)
    
    # Background gradient
    for i in range(size):
        img[i, :, 0] = int(255 * i / size)  # Red gradient top-bottom
        img[:, i, 2] = int(255 * i / size)  # Blue gradient left-right
    
    # Add a white circle
    center = size // 2
    radius = size // 4
    for i in range(size):
        for j in range(size):
            if (i - center)**2 + (j - center)**2 < radius**2:
                img[i, j] = [255, 255, 255]
    
    return img

# Create and display
sample_img = create_sample_image(64)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(sample_img)
axes[0].set_title(f'Full Color Image\nShape: {sample_img.shape}')

axes[1].imshow(sample_img[:,:,0], cmap='Reds')
axes[1].set_title('Red Channel (R)')

axes[2].imshow(sample_img[:,:,1], cmap='Greens')
axes[2].set_title('Green Channel (G)')

axes[3].imshow(sample_img[:,:,2], cmap='Blues')
axes[3].set_title('Blue Channel (B)')

for ax in axes:
    ax.axis('off')

plt.suptitle('Image = 3D Tensor (Height × Width × Channels)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('rgb_channels.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📊 Image Properties:")
print(f"   Shape: {sample_img.shape} → (Height={sample_img.shape[0]}, Width={sample_img.shape[1]}, Channels={sample_img.shape[2]})")
print(f"   Data type: {sample_img.dtype}")
print(f"   Value range: [{sample_img.min()}, {sample_img.max()}]")
print(f"   Total pixels: {sample_img.shape[0] * sample_img.shape[1]}")
print(f"   Total values: {sample_img.size}")
print(f"   Memory: {sample_img.nbytes} bytes")

## Image Quality Metrics — Mathematical Foundations for RL Rewards

For RL agents processing images, the reward function determines what the agent learns to optimize. Here we derive the key quality metrics used as RL rewards.

### PSNR (Peak Signal-to-Noise Ratio)

$$\text{PSNR}(I_1, I_2) = 10 \log_{10}\frac{L^2}{\text{MSE}} = 20\log_{10}\frac{L}{\sqrt{\text{MSE}}} \quad \text{(dB)}$$

where $L$ is the maximum pixel value (255 for 8-bit) and $\text{MSE} = \frac{1}{N}\sum_{i}(I_1[i] - I_2[i])^2$.

**Interpretation scale:**
- PSNR > 40 dB: Excellent (differences imperceptible)
- 30-40 dB: Good (minor visible differences)
- 20-30 dB: Poor (significant degradation)
- < 20 dB: Very poor

**Limitation:** PSNR treats all pixel errors equally, regardless of whether they occur in perceptually important regions.

### SSIM (Structural Similarity Index)

$$\text{SSIM}(x, y) = l(x,y)^{\alpha} \cdot c(x,y)^{\beta} \cdot s(x,y)^{\gamma}$$

with $\alpha = \beta = \gamma = 1$:

$$= \frac{(2\mu_x\mu_y + C_1)(2\sigma_{xy} + C_2)}{(\mu_x^2 + \mu_y^2 + C_1)(\sigma_x^2 + \sigma_y^2 + C_2)}$$

**Stability constants:** $C_1 = (K_1 L)^2, C_2 = (K_2 L)^2$ with $K_1 = 0.01, K_2 = 0.03$.

**Key property:** SSIM $\in [-1, 1]$ with SSIM $= 1$ iff $x = y$.

**Proof that SSIM = 1 iff identical:** If $x = y$: $\mu_x = \mu_y$, $\sigma_x = \sigma_y$, $\sigma_{xy} = \sigma_x^2$. Then numerator $= (2\mu_x^2 + C_1)(2\sigma_x^2 + C_2) =$ denominator. $\blacksquare$

### Composite RL Reward

A practical RL reward combines multiple metrics:
$$R = w_1 \cdot \text{PSNR} + w_2 \cdot \text{SSIM} - w_3 \cdot \text{LPIPS}$$

where LPIPS (Learned Perceptual Image Patch Similarity) uses deep features for perceptual comparison. The weights $w_1, w_2, w_3$ control the tradeoff between pixel accuracy, structural preservation, and perceptual quality.

In [ ]:
# Zoom into a small patch to see actual pixel values
patch = sample_img[28:36, 28:36]  # 8x8 patch near center

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Show the patch location on original
img_annotated = sample_img.copy()
cv2.rectangle(img_annotated, (28, 28), (36, 36), (0, 255, 0), 1)
axes[0].imshow(img_annotated)
axes[0].set_title('Patch Location (green box)')

# Show zoomed patch with pixel values
for ch_idx, (ch_name, cmap) in enumerate(zip(['Red', 'Green', 'Blue'], ['Reds', 'Greens', 'Blues'])):
    ax = axes[ch_idx + 1]
    channel_patch = patch[:, :, ch_idx]
    ax.imshow(channel_patch, cmap=cmap, vmin=0, vmax=255)
    
    # Annotate with actual values
    for i in range(8):
        for j in range(8):
            val = channel_patch[i, j]
            color = 'white' if val < 128 else 'black'
            ax.text(j, i, str(val), ha='center', va='center', 
                   fontsize=6, color=color)
    ax.set_title(f'{ch_name} Channel Values')

for ax in axes:
    ax.axis('off')

plt.suptitle('Zoomed 8×8 Patch — Every Pixel is Just a Number!', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('pixel_values.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Why Images Are Perfect for RL

### The RL-Image Connection:

| RL Concept | Image Processing Equivalent |
|:-----------|:---------------------------|
| **State** $s$ | Current image pixel values $\mathbf{I} \in \mathbb{R}^{H \times W \times C}$ |
| **Action** $a$ | Image operation (brighten, blur, crop, etc.) |
| **Reward** $r$ | Image quality metric (PSNR, SSIM, aesthetic score) |
| **Policy** $\pi(a|s)$ | Decision of which operation to apply |
| **Transition** $P(s'|s,a)$ | Result of applying operation to image |

### Key Insight:
$$\text{Image Processing} = \text{Sequential Decision Making}$$

Every image enhancement pipeline is a sequence of decisions — which filter to apply, how much to adjust contrast, where to crop. **RL learns this sequence optimally!**

In [ ]:
# Simple demonstration: Image as RL state, operations as actions
np.random.seed(42)

# Start with a "degraded" image (our initial state)
original = create_sample_image(64).astype(np.float32) / 255.0
degraded = original + np.random.normal(0, 0.1, original.shape)  # Add noise
degraded = np.clip(degraded, 0, 1)

# Define simple "actions" an RL agent could take
def action_blur(img, kernel_size=3):
    """Action: Apply Gaussian blur"""
    return cv2.GaussianBlur(img, (kernel_size, kernel_size), 0)

def action_brighten(img, factor=1.2):
    """Action: Increase brightness"""
    return np.clip(img * factor, 0, 1)

def action_sharpen(img):
    """Action: Sharpen image"""
    kernel = np.array([[-1,-1,-1], [-1,9,-1], [-1,-1,-1]]) / 9.0
    return np.clip(cv2.filter2D(img, -1, kernel), 0, 1)

# Reward function: how close to original
def reward_psnr(img, target):
    """Reward: Peak Signal-to-Noise Ratio"""
    mse = np.mean((img - target)**2)
    if mse == 0:
        return 100.0
    return 10 * np.log10(1.0 / mse)

# Show the RL concept
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

states = [degraded]
actions_taken = ['Start (Noisy)']
rewards = [reward_psnr(degraded, original)]

# Step 1: Blur (denoise)
state_1 = action_blur(degraded, 3)
states.append(state_1)
actions_taken.append('Action: Blur (denoise)')
rewards.append(reward_psnr(state_1, original))

# Step 2: Sharpen
state_2 = action_sharpen(state_1)
states.append(state_2)
actions_taken.append('Action: Sharpen')
rewards.append(reward_psnr(state_2, original))

for idx, (state, action, reward) in enumerate(zip(states, actions_taken, rewards)):
    row = idx // 3
    col = idx % 3
    axes[row, col].imshow(np.clip(state, 0, 1))
    axes[row, col].set_title(f'Step {idx}: {action}\nReward (PSNR): {reward:.2f} dB')
    axes[row, col].axis('off')

# Show original target
axes[1, 0].imshow(original)
axes[1, 0].set_title('🎯 Target (Original)')
axes[1, 0].axis('off')

# Reward plot
axes[1, 1].bar(range(len(rewards)), rewards, color=['red', 'orange', 'green'])
axes[1, 1].set_xlabel('Step')
axes[1, 1].set_ylabel('PSNR (dB)')
axes[1, 1].set_title('Reward Improves with Actions!')
axes[1, 1].set_xticks(range(len(rewards)))

axes[1, 2].text(0.5, 0.5, 'RL Goal:\nFind BEST sequence\nof actions to maximize\ncumulative reward!',
               ha='center', va='center', fontsize=14,
               bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8),
               transform=axes[1, 2].transAxes)
axes[1, 2].axis('off')

plt.suptitle('🤖 RL for Image Processing — The Big Picture', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('rl_image_concept.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Summary & Key Takeaways

### What we learned:
1. **An image is a function** $f(x,y)$ mapping spatial coordinates to intensity
2. **Digital images** are sampled (discretized in space) and quantized (discretized in intensity)
3. **Image = Matrix** of shape $(H, W)$ for grayscale or $(H, W, C)$ for color
4. **Each pixel** is just a number (0-255 for 8-bit images)
5. **RL connection**: Image is the state, operations are actions, quality metrics are rewards

### Mathematical Summary:
$$\text{Continuous: } f: \mathbb{R}^2 \rightarrow \mathbb{R}$$
$$\text{Digital: } \mathbf{I}: \{0,...,M-1\} \times \{0,...,N-1\} \rightarrow \{0,...,L-1\}$$
$$\text{Color: } \mathbf{I} \in \mathbb{R}^{M \times N \times C}$$

---
**Next:** Module 1.2 — Pixels and Channels (Deep Dive)